In [ ]:
import joblib
import pandas as pd
import numpy as np
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
from kagglehub import KaggleDatasetAdapter

%load_ext autoreload
%autoreload 2

Ensemble Methods, bag, boost stack and vote

In [ ]:
file_path = "ecommerce_user_behavior_8000.csv"

# Load the latest version
raw_df = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "asifxzaman/e-commerce-behavior-dataset8000-users",
  file_path,
)

In [ ]:
raw_df.isna().sum()

In [ ]:
raw_df.duplicated(subset='user_id').sum(), raw_df[raw_df.duplicated()].shape , raw_df[raw_df['age'].isna()].shape, raw_df[raw_df['device_type'].isna()].shape

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6), nrows = 2, ncols = 3)
ax = ax.ravel()

sns.boxplot(x='gender', y='time_on_site', hue='device_type', data=raw_df, ax=ax[0], legend=False)
ax[0].set_title('Time on Site')
sns.boxplot(x='gender', y='pages_viewed', hue='device_type', data=raw_df, ax=ax[1], legend=False)
ax[1].set_title('Pages Viewed')
sns.boxplot(x='gender', y='previous_purchases', hue='device_type', data=raw_df, ax=ax[2], legend=False)
ax[2].set_title('Previous Purchases')
sns.boxplot(x='gender', y='cart_items', hue='device_type', data=raw_df, ax=ax[3], legend=False)
ax[3].set_title('Cart Items')
sns.boxplot(x='gender', y='avg_session_time', hue='device_type', data=raw_df, ax=ax[4], legend=False)
ax[4].set_title('Average Session Time')
sns.boxplot(x='gender', y='bounce_rate', hue='device_type', data=raw_df, ax=ax[5], legend=False)
ax[5].set_title('Bounce Rate')
fig.legend(bbox_to_anchor=(1.05, 1.0), loc='upper right')
plt.tight_layout()

In [ ]:
did_not_purchase = (raw_df['user_id'].nunique() - raw_df['purchase'].sum()).astype(int)
f'{did_not_purchase} users did not purchase'

In [ ]:
raw_df.groupby(['gender', 'device_type']).agg(**{
    'Total Users': ('user_id', 'nunique'),
    'Purchases': ('purchase', 'sum'),
    'Total Cart Items': ('cart_items', 'sum'),
    'Discounts Seen': ('discount_seen', 'sum'),
    'Ad Clicks': ('ad_clicked', 'sum'),
    'Returning Users': ('returning_user', 'sum')
})

In [ ]:
to_filter_rows = (raw_df['age'].isna() | raw_df['device_type'].isna() | raw_df['discount_seen'].isna() | raw_df['ad_clicked'].isna() | raw_df['returning_user'].isna() | raw_df['purchase'].isna())
df = raw_df[~to_filter_rows].copy()
df['user_id'] = df['user_id'].fillna(0).astype(int)
df['gender'] = df['gender'].fillna('Unknown')
fill_in_columns = ['time_on_site', 'pages_viewed', 'previous_purchases', 'cart_items', 'avg_session_time', 'bounce_rate']
df[fill_in_columns] = df.groupby(['age', 'device_type'])[fill_in_columns].transform(lambda x: x.fillna(x.mean()))
df[['discount_seen', 'ad_clicked', 'returning_user', 'purchase']] =df[['discount_seen', 'ad_clicked', 'returning_user', 'purchase']].astype(bool)
f'Records dropped: {raw_df.shape[0] - df.shape[0]}'

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6), nrows = 2, ncols = 3)
ax = ax.ravel()

sns.boxplot(x='gender', y='time_on_site', hue='purchase', data=df, ax=ax[0], legend=False)
ax[0].set_title('Time on Site')
sns.boxplot(x='gender', y='pages_viewed', hue='purchase', data=df, ax=ax[1], legend=False)
ax[1].set_title('Pages Viewed')
sns.boxplot(x='gender', y='previous_purchases', hue='purchase', data=df, ax=ax[2], legend=False)
ax[2].set_title('Previous Purchases')
sns.boxplot(x='gender', y='cart_items', hue='purchase', data=df, ax=ax[3], legend=False)
ax[3].set_title('Cart Items')
sns.boxplot(x='gender', y='avg_session_time', hue='purchase', data=df, ax=ax[4], legend=False)
ax[4].set_title('Average Session Time')
sns.boxplot(x='gender', y='bounce_rate', hue='purchase', data=df, ax=ax[5], legend=False)
ax[5].set_title('Bounce Rate')
fig.legend(bbox_to_anchor=(1.05, 1.0), loc='upper right')
plt.tight_layout()